# [3] Naive Trial with RoBERTa

## Imports

In [11]:
import env

In [33]:
from epidec.datasets import SWUnivDaconDataset

from transformers import pipeline, AutoTokenizer
from torch.utils.data import DataLoader

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import json
import sys

## Load Datasets

In [13]:
DATA_ROOT = "./data"

train_dataset = SWUnivDaconDataset(DATA_ROOT, train=True, valid_ratio=0.1)
valid_dataset = SWUnivDaconDataset(DATA_ROOT, valid=True, valid_ratio=0.1)
test_dataset = SWUnivDaconDataset(DATA_ROOT, train=False)

print(f"INFO: Dataset loaded successfully. Train - {len(train_dataset)}, Valid - {len(valid_dataset)}, Test - {len(test_dataset)}")

INFO: Dataset 'swuniv_dacon' already exists at data. Skipping download.
INFO: Dataset 'swuniv_dacon' already exists at data. Skipping download.
INFO: Dataset 'swuniv_dacon' already exists at data. Skipping download.
INFO: Dataset loaded successfully. Train - 87454, Valid - 9718, Test - 1962


In [14]:
train_dataset[1]

("장 담그기는 대한민국의 국가무형문화재 제137호이다. 장(醬)은 고추장, 된장, 간장 등 발효된 짠 조미료를 통틀어 이르는 말한다. \n 2018년 11월 1일 문화재청장이 문화재 지정을 예고하였으며, 2018년 12월 27일 문화재로 지정되었음이 2019년 1월 9일 관보에 고시되었다. 보유자나 보유단체를 인정하지 않고, 종목만 지정된 국가 무형 문화재이다. \n 장(醬)은 콩을 재료로 하여 소금에 버무려 만든 양념의 일종으로 오랜 세월 동안 한국인의 밥상에서 중요한 위치를 차지하고 있고, 우리 음식의 근간이다. \n 장 담그기에 대한 기록은 삼국시대 이전부터 찾아 볼 수 있고, 겨울 김장과 더불어 가족공동체의 연례행사로서 행위 그 자체가 우리 고유의 문화적 정체성을 형성하는 데 중요한 역할을 한다. \n 장 담그기는 각 가정에서 여성들에 의해 구전(口傳)을 통해 전승되고 있으며, 한국의 주거문화, 세시풍속, 기복신앙, 전통과학적 요소를 복합적으로 반영하고 있는 문화적 자산이다. \n 우리나라의 장 담그기는 여러 종류의 장 명칭에 ‘장’이라는 글자가 동일하게 들어간다는 점, 연초(年初)에 1회의 장 담그는 과정을 거친 후 된장과 간장 두 가지의 장을 만든다는 점, 전년도에 쓰고 남은 씨간장을 이용하여 수년 동안 겹간장의 형식을 거친다는 점은 동남아시아 '두장(豆醬) 문화권' 내에서도 한국 장 담그기가 갖는 특징이자 독창적인 대목이다. \n 이처럼 장 담그기는 무형문화재로서 역사성, 예술성, 학술성 등의 가치가 있으므로 이를 국가무형문화재로 지정하여 종목을 보존 전승하고자 함. 다만, 장 담그기는 특정지역에 한정되어 전승되는 전통 지식, 기술이 아니므로 보유자 또는 보유단체를 인정하지 않고 종목으로만 지정한다.",
 0)

In [15]:
valid_dataset[1]

('《인간사화》는 왕궈웨이가 1910년에 남긴 평론이다. \n 사화는 사 창작의 본질, 수사, 사 작가와 작품 및 시대에 대한 평가, 사 작가들의 행적이나 창작 배경 등을 논하는 것이다. 일찍이 서양의 철학과 미학, 문예 사상 등을 접했던 왕궈웨이는 사 창작을 하면서 중국과 서양의 문예 미학을 융합해 참신한 문예 미학관을 정립하고, 새로운 문예비평의 패러다임을 세웠다. 이 결과물이 ≪인간사화≫로 ‘경계(境界)’를 문예 미학의 최고 이상으로 내세우고 있다. \n ≪인간사화≫는 원래 왕궈웨이가 생전에 친히 64조목으로 편정하여, ＜국수학보(國粹學報)＞에 발표했는데 후인들이 ≪인간사화≫를 연구하고 주소하는 과정에서 유고를 수집해 증보하여 권1 ≪인간사화≫ 64조목, 권2 ≪인간사화 미간고(未刊稿)≫ 50조목, 권3 ≪인간사화 산고(刪稿)≫ 13조목, 권4 ≪인간사화 부록(附錄)≫ 28조목의 체제를 구성했다. 판본에 따라 권수나 조목 수가 다소 차이가 있기는 하나 왕궈웨이의 원의에 가장 부합하는 체제로 평가받는 삼민서국본의 편차로, 이를 번역의 텍스트로 삼아 ≪인간사화≫ 전체 내용을 살펴볼 수 있다. \n 왕궈웨이는 ‘경계’를 문학의 최고 이상으로 여겨 문학의 예술 미학을 추구했으면서도, 또한 인간의 삶과 인생 근원 문제를 둘러싸고 사유를 진행하여 인간의 완미한 경지를 추구하는 인간 미학을 탐구했다는 점에서 높이 평가받는다. 또한 그의 경계설은 중·서 문예 미학의 결합이라는 점에서 의의가 있다. ‘경계’설은 중국 전통의 천인합일·물아일체 등의 관념, 노장(老莊) 등의 무위자연 철학 사상, 중국 전통의 미학·문예 사상 등에 연원을 두고, 서양의 칸트, 쇼펜하우어, 니체 등의 철학, 미학, 문예 사상의 영양을 섭취하고 각종 미학, 철학 개념을 빌려 이루어졌다.',
 0)

## Define Model

In [16]:
model_id = "roberta-large-openai-detector"

pipe = pipeline("text-classification", model=model_id)
tokenizer = AutoTokenizer.from_pretrained(model_id)

Some weights of the model checkpoint at roberta-large-openai-detector were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cuda:0


In [23]:
def chunk_text(text, tokenizer=tokenizer, max_length=400):
    words = text.split()
    chunks = []
    current_chunk = []
    current_length = 0

    for word in words:
        word_tokens = len(tokenizer.encode(word, add_special_tokens=False))
        if current_length + word_tokens > max_length and current_chunk:
            chunks.append(" ".join(current_chunk))
            current_chunk = [word]
            current_length = word_tokens
        else:
            current_chunk.append(word)
            current_length += word_tokens

    if current_chunk:
        chunks.append(" ".join(current_chunk))

    return chunks

## Evaluation

In [44]:
check_only_for = None
threshold = 0.7
mini_batch_size = 4

In [46]:
# Validation
corrects, errors, true_human, false_human, results = [], [], [], [], []
progress = tqdm(DataLoader(valid_dataset, batch_size=1, shuffle=True), desc="Validating...")

for idx, data in enumerate(progress):
    label = data[1][0]
    data = data[0][0]

    if check_only_for is not None and label != check_only_for:
        continue  # Skip if the label does not match the specified check

    chunks = chunk_text(data)
    preds = [threshold, threshold]
    preds_all = []

    for chunk in chunks:
        pred = pipe(chunk)[0]['score']
        preds_all.append(pred)
        if pred < preds[0]:
            preds[0] = pred
        elif pred > preds[1]:
            preds[1] = pred

    if preds[0] >= threshold and preds[1] >= threshold:
        predicted = 1
    elif preds[0] < threshold and preds[1] < threshold:
        predicted = 0
    else:
        predicted = np.mean(preds_all)

    result = dict(question=data, label=label, predicted=predicted)
    predicted_label = 1 if predicted >= threshold else 0
    if predicted_label == label:
        corrects.append(result)
        if label == 0:
            true_human.append(result)
    else:
        errors.append(result)
        print(f"ERROR: Incorrect prediction for index {idx}, expected {label}, got {predicted}")
        if label == 0:
            false_human.append(result)
    results.append(result)
    progress.set_description(f"Correct: {len(corrects)}/{len(results)} [H: {len(true_human)}, A: {len(corrects)-len(true_human)}], Errors: {len(errors)}/{len(results)} [H: {len(false_human)}, A: {len(errors)-len(false_human)}]")

print(f"INFO: Correct: {len(corrects)}/{len(results)}, Errors: {len(errors)}/{len(results)}")

Validating...:   0%|          | 0/9718 [00:00<?, ?it/s]

ERROR: Incorrect prediction for index 0, expected 0, got 0.7793028593063355
ERROR: Incorrect prediction for index 7, expected 0, got 0.7640253901481628
ERROR: Incorrect prediction for index 9, expected 1, got 0.5917814870675405
ERROR: Incorrect prediction for index 14, expected 1, got 0.5900620281696319
ERROR: Incorrect prediction for index 16, expected 0, got 0.7259490660258702
ERROR: Incorrect prediction for index 25, expected 1, got 0.6280654966831207
ERROR: Incorrect prediction for index 36, expected 0, got 0.7503321230411529
ERROR: Incorrect prediction for index 46, expected 1, got 0.5679457426071167
ERROR: Incorrect prediction for index 48, expected 1, got 0.5907889397247977
ERROR: Incorrect prediction for index 52, expected 0, got 0.7717694938182831
ERROR: Incorrect prediction for index 54, expected 0, got 0.7848386317491531
ERROR: Incorrect prediction for index 57, expected 1, got 0.5894264101982116
ERROR: Incorrect prediction for index 59, expected 0, got 0.7386200753125277
ER

KeyboardInterrupt: 

In [47]:
# Test
results = []
for idx, data in enumerate(tqdm(test_dataset, desc="Testing...")):
    data = data[0]
    chunks = chunk_text(data)
    preds = [threshold, threshold]
    preds_all = []

    for chunk in chunks:
        pred = pipe(chunk)[0]['score']
        preds_all.append(pred)
        if pred < preds[0]:
            preds[0] = pred
        elif pred > preds[1]:
            preds[1] = pred

    if preds[0] >= threshold and preds[1] >= threshold:
        predicted = 1
    elif preds[0] < threshold and preds[1] < threshold:
        predicted = 0
    else:
        predicted = np.mean(preds_all)

    result = dict(question=data, label=predicted)
    results.append(result)

Testing...:   0%|          | 0/1962 [00:00<?, ?it/s]

In [58]:
r = [1 if results['label'] >= threshold else 0 for results in results]

In [59]:
sub = pd.read_csv("./data/swuniv_dacon/" + test_dataset.submission_file, encoding='utf-8-sig')

In [60]:
sub

,ID,generated
0,TEST_0000,0
1,TEST_0001,0
2,TEST_0002,0
3,TEST_0003,0
4,TEST_0004,0
...,...,...
1957,TEST_1957,0
1958,TEST_1958,0
1959,TEST_1959,0
1960,TEST_1960,0


In [61]:
sub['generated'] = r

In [62]:
sub

,ID,generated
0,TEST_0000,0
1,TEST_0001,0
2,TEST_0002,0
3,TEST_0003,0
4,TEST_0004,0
...,...,...
1957,TEST_1957,0
1958,TEST_1958,0
1959,TEST_1959,0
1960,TEST_1960,0


In [64]:
sub.to_csv("./data/submission_roberta.csv", index=False, encoding='utf-8-sig')